In [3]:
import os
import imageio
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# Configuration
repo_id = "jogarulfop/2026-07-28_shake4it_bench_dragonfly_10kHz_nfft_512"
episodes_to_extract = [0, 10, 20, 30]
save_dir = "data/"
os.makedirs(save_dir, exist_ok=True)

# 1. Load the dataset
dataset = LeRobotDataset(repo_id)
fps = dataset.fps

# 2. Get all available camera/video keys dynamically by checking feature names
camera_keys = [key for key in dataset.features.keys() if "image" in key]
print(f"Found {len(camera_keys)} camera stream(s): {camera_keys}")

for ep_idx in episodes_to_extract:
    # Find the frame indices belonging to this specific episode
    ep_indices = [i for i, ep in enumerate(dataset.hf_dataset["episode_index"]) if ep == ep_idx]
    
    if not ep_indices:
        print(f"Episode {ep_idx} not found in the dataset.")
        continue
        
    print(f"\nExtracting {len(ep_indices)} frames for Episode {ep_idx}...")
    
    # 3. Loop through EVERY camera key found in the dataset
    for cam_key in camera_keys:
        frames = []
        
        for i in ep_indices:
            # dataset[i] returns PyTorch tensors [C, H, W] scaled from 0.0 to 1.0
            frame_tensor = dataset[i][cam_key]
            
            # Convert tensor to numpy array [H, W, C] scaled from 0 to 255
            img_np = (frame_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
            frames.append(img_np)
        
        # Format the file name using the last part of the camera key (e.g., 'top', 'wrist')
        # observation.images.top -> top
        cam_name_short = cam_key.split('.')[-1]
        
        # 4. Re-encode and save as an MP4 file
        out_path = os.path.join(save_dir, f"episode_{ep_idx:06d}_{cam_name_short}.mp4")
        imageio.mimwrite(out_path, frames, fps=fps, codec="libx264")
        print(f"Saved: {out_path}")

Found 3 camera stream(s): ['observation.images.top', 'observation.images.wrist', 'observation.images.tactile_spectrogram_dragonfly_10kHz_nfft_512']

Extracting 738 frames for Episode 0...
Saved: data/episode_000000_top.mp4
Saved: data/episode_000000_wrist.mp4
Saved: data/episode_000000_tactile_spectrogram_dragonfly_10kHz_nfft_512.mp4

Extracting 725 frames for Episode 10...
Saved: data/episode_000010_top.mp4
Saved: data/episode_000010_wrist.mp4
Saved: data/episode_000010_tactile_spectrogram_dragonfly_10kHz_nfft_512.mp4

Extracting 704 frames for Episode 20...
Saved: data/episode_000020_top.mp4
Saved: data/episode_000020_wrist.mp4
Saved: data/episode_000020_tactile_spectrogram_dragonfly_10kHz_nfft_512.mp4

Extracting 974 frames for Episode 30...
Saved: data/episode_000030_top.mp4
Saved: data/episode_000030_wrist.mp4
Saved: data/episode_000030_tactile_spectrogram_dragonfly_10kHz_nfft_512.mp4
